# Phase 6B — P1 controlled one-hour training pilot

**Prepared; not run.** Two private Kaggle inputs: `phase6b_pilot_train_bundle` (581 frozen-training clips, 1.002867 h) and the existing `phase5_validation_bundle` (732 frozen-validation clips, 1.881808 h). Never attach test. Pinned Base starts from pretrained weights, **not** Phase 6A's updated checkpoint. Three epoch maximum, validation CER after each epoch, checkpoint by minimum validation corpus CER, and full private predictions. This is a small pilot; full F1/LoRA experiments remain later. Run the setup cell first; if it installs packages, restart Kaggle and Run All. Use Tesla T4 + Internet. Keep all inputs/outputs private.


In [ ]:
import importlib.metadata as im, subprocess, sys
PINS={'transformers':'4.57.1','tokenizers':'0.22.2','huggingface-hub':'0.36.2','safetensors':'0.8.0'}
def version(name):
    try:return im.version(name)
    except im.PackageNotFoundError:return None
current={k:version(k) for k in PINS}
print('Installed:',current)
if current!=PINS:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--no-deps',*[f'{k}=={v}' for k,v in PINS.items()]],check=True)
    raise SystemExit('Pinned Hugging Face packages installed. Restart Kaggle session, then Run All. PyTorch/CUDA unchanged.')
if 'transformers' in sys.modules and getattr(sys.modules['transformers'],'__version__',None)!='4.57.1':
    raise SystemExit('Restart Kaggle: incompatible Transformers already imported.')


## 1. Runtime and exact private bundles

Both bundles are identified by their metadata; all payload SHA-256 checks run before model loading. The training bundle is one deterministic nested one-hour subset of the frozen train manifest. Validation membership and B0 decoding input are unchanged.


In [ ]:
import copy,csv,hashlib,json,math,os,platform,random,re,time,unicodedata,zipfile
from pathlib import Path
from io import BytesIO
import numpy as np, soundfile as sf, torch, transformers, tokenizers, huggingface_hub, safetensors
from transformers import WhisperProcessor,WhisperForConditionalGeneration,GenerationConfig,get_linear_schedule_with_warmup
assert torch.__version__.startswith('2.11.0') and torch.cuda.is_available()
assert 'T4' in torch.cuda.get_device_name(0)
assert transformers.__version__=='4.57.1' and tokenizers.__version__=='0.22.2'
assert huggingface_hub.__version__=='0.36.2' and safetensors.__version__=='0.8.0'
DEVICE=torch.device('cuda:0');SEED=0
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
print({'python':platform.python_version(),'torch':torch.__version__,'transformers':transformers.__version__,'gpu':torch.cuda.get_device_name(0)})
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def bundle(kind):
    matches=[]
    for marker in Path('/kaggle/input').rglob('bundle_record.json'):
        try:r=json.loads(marker.read_text())
        except (OSError,ValueError):continue
        if r.get('bundle_type')==kind:matches.append((marker.parent.resolve(),r))
    if not matches:
        for archive in Path('/kaggle/input').rglob('*.zip'):
            with zipfile.ZipFile(archive) as z:
                markers=[name for name in z.namelist() if name.endswith('bundle_record.json')]
                for name in markers:
                    record=json.loads(z.read(name))
                    if record.get('bundle_type')!=kind:continue
                    target=Path('/kaggle/working/phase6b_extracted_inputs')/kind
                    target.mkdir(parents=True,exist_ok=True)
                    for member in z.infolist():
                        path=(target/member.filename).resolve()
                        path.relative_to(target.resolve())
                    z.extractall(target)
                    matches.append(((target/Path(name).parent).resolve(),record))
    assert len(matches)==1,f'Expected one {kind} input, found {len(matches)}'
    return matches[0]
TRAIN_ROOT,TRAIN_RECORD=bundle('phase6b_pilot_train_v1')
VAL_ROOT,VAL_RECORD=bundle('phase5_validation_baseline')
assert TRAIN_RECORD['contains_validation_or_test_audio'] is False
assert VAL_RECORD['evaluation_partition']=='validation'
assert VAL_RECORD['contains_train_audio'] is False and VAL_RECORD['contains_test_audio'] is False
def safe(root,relative):
    path=(root/relative).resolve();path.relative_to(root);return path
def verify_sums(root):
    entries=[line.split('  ',1) for line in (root/'SHA256SUMS.txt').read_text().splitlines() if line]
    assert len(entries)==len({path for _,path in entries})
    for expected,relative in entries:assert sha(safe(root,relative))==expected,f'Bundle checksum failed: {relative}'
    return len(entries)
assert verify_sums(TRAIN_ROOT)==TRAIN_RECORD['clips']+2
assert verify_sums(VAL_ROOT)==737
assert sha(TRAIN_ROOT/'train_manifest.csv')==TRAIN_RECORD['manifest_sha256']
assert sha(VAL_ROOT/'validation_manifest.csv')==VAL_RECORD['manifest_sha256']
with (TRAIN_ROOT/'train_manifest.csv').open(newline='',encoding='utf-8') as f:train=list(csv.DictReader(f))
with (VAL_ROOT/'validation_manifest.csv').open(newline='',encoding='utf-8') as f:validation=list(csv.DictReader(f))
assert len(train)==TRAIN_RECORD['clips']==581 and len(validation)==732
assert len({r['sample_id'] for r in train})==581 and len({r['component_id'] for r in train})==15
assert len({r['sample_id'] for r in validation})==732 and len({r['component_id'] for r in validation})==16
assert set(r['sample_id'] for r in train).isdisjoint(r['sample_id'] for r in validation)
assert all(r['split']=='train' for r in train) and all(r['split']=='validation' for r in validation)
assert hashlib.sha256(('\n'.join(sorted(r['sample_id'] for r in train))+'\n').encode()).hexdigest()==TRAIN_RECORD['train_membership_sha256']
assert sum(len(r['reference']) for r in validation)==64137
print('Verified private train/validation bundles:',len(train),len(validation),'clips; no test data.')


## 2. Pinned model, target policy and B0 generation policy

Training target: NFD IPA text tokens plus one EOS; model inserts decoder start. No invented Nyishi token or forced English token. Validation generation uses the **Phase 5B bundle's exact saved generation config**, explicitly resetting `forced_decoder_ids=None`, language/task=None, greedy FP32, maximum 448 and inherited begin suppression. This prevents the saved model default from silently changing the B0 comparison.


In [ ]:
MODEL_ID='neurlang/ipa-whisper-base'; REVISION='949f7c4e4ee374b5202bc5886f6e4552e5ae2371'
processor=WhisperProcessor.from_pretrained(MODEL_ID,revision=REVISION)
GEN=json.loads((VAL_ROOT/'generation_config.json').read_text())
generation_config=GenerationConfig.from_dict(GEN)
generation_config.forced_decoder_ids=None;generation_config.language=None;generation_config.task=None
generation_config.suppress_tokens=[];generation_config.do_sample=False;generation_config.num_beams=1
generation_config.return_dict_in_generate=True
assert generation_config.max_length==448 and generation_config.begin_suppress_tokens==[220,50257]
def normalized(value):return re.sub(r' {2,}',' ',unicodedata.normalize('NFD',str(value)).strip())
def read_audio(root,row):
    path=safe(root,row['audio_file']);assert sha(path)==row['audio_sha256']
    data,sr=sf.read(path,dtype='float32',always_2d=True)
    assert sr==16000 and data.shape[1]==1 and 0.5<=len(data)/sr<30
    assert abs(len(data)/sr-float(row['duration_seconds']))<0.02
    return data[:,0]
# Features in host RAM; ~0.56 GiB for 581 fixed 80x3000 arrays. Validation streams from disk.
train_features=[];train_targets=[]
for row in train:
    assert normalized(row['reference_raw'])==row['reference']==normalized(row['reference'])
    feature=processor.feature_extractor(read_audio(TRAIN_ROOT,row),sampling_rate=16000,return_tensors='pt').input_features[0].numpy()
    assert feature.shape==(80,3000) and np.isfinite(feature).all()
    ids=processor.tokenizer(row['reference'],add_special_tokens=False).input_ids
    assert processor.tokenizer.decode(ids,skip_special_tokens=False)==row['reference']
    target=ids+[processor.tokenizer.eos_token_id]
    assert len(target)+1<=448
    train_features.append(feature);train_targets.append(target)
print('Prepared',len(train_features),'training features; target tokens min/max',min(map(len,train_targets)),max(map(len,train_targets)))


## 3. Training configuration and checkpoint resume

Three maximum epochs, AdamW 1e-5, weight decay 0.01, linear schedule with 5% warmup, clip norm 1.0, seed 0, micro-batch 2 / accumulation 4 (fallback 1/8 only after measured OOM). FP16 AMP with FP32 weights and optimizer, SDPA, gradient checkpointing off initially. Every completed epoch saves full training state and evaluates all 732 validation clips. Set `RESUME_CHECKPOINT` to a private saved epoch directory only when resuming the *same* run.


In [ ]:
import gc
OUT=Path('/kaggle/working/phase6b_P1_private');OUT.mkdir(exist_ok=True)
RESUME_CHECKPOINT=None  # Example: Path('/kaggle/input/private-p1-checkpoint/epoch_01')
MICRO_BATCH=2;ACCUMULATION=4;MAX_EPOCHS=3
assert MICRO_BATCH*ACCUMULATION==8
steps_per_epoch=math.ceil(math.ceil(len(train)/MICRO_BATCH)/ACCUMULATION)
total_steps=steps_per_epoch*MAX_EPOCHS;warmup_steps=max(1,math.ceil(.05*total_steps))
model_source=RESUME_CHECKPOINT/'model' if RESUME_CHECKPOINT else MODEL_ID
model=WhisperForConditionalGeneration.from_pretrained(model_source,**({} if RESUME_CHECKPOINT else {'revision':REVISION}),dtype=torch.float32,attn_implementation='sdpa').to(DEVICE)
model.config.use_cache=False
assert next(model.parameters()).dtype==torch.float32
decay=[];no_decay=[]
for name,param in model.named_parameters():
    if not param.requires_grad:continue
    (no_decay if name.endswith('bias') or 'layer_norm' in name.lower() else decay).append(param)
assert decay and no_decay
optimizer=torch.optim.AdamW([{'params':decay,'weight_decay':.01},{'params':no_decay,'weight_decay':0.0}],lr=1e-5)
print({'decay_parameters':sum(p.numel() for p in decay),'no_decay_parameters':sum(p.numel() for p in no_decay)})
scheduler=get_linear_schedule_with_warmup(optimizer,warmup_steps,total_steps)
scaler=torch.amp.GradScaler('cuda',init_scale=256.0)
start_epoch=0;history=[];best_cer=float('inf');best_epoch=None;completed_steps=0
fingerprint=hashlib.sha256(json.dumps({'model':MODEL_ID,'revision':REVISION,'train':TRAIN_RECORD['train_membership_sha256'],
 'validation':VAL_RECORD['manifest_sha256'],'seed':SEED,'micro_batch':MICRO_BATCH,'accumulation':ACCUMULATION,
 'max_epochs':MAX_EPOCHS,'generation':generation_config.to_dict()},sort_keys=True,default=str).encode()).hexdigest()
if RESUME_CHECKPOINT:
    saved=torch.load(RESUME_CHECKPOINT/'training_state.pt',map_location='cpu',weights_only=False)
    assert saved['fingerprint']==fingerprint and saved['epoch']<MAX_EPOCHS
    optimizer.load_state_dict(saved['optimizer'])
    for item in optimizer.state.values():
        for key,val in item.items():
            if isinstance(val,torch.Tensor):item[key]=val.to(DEVICE)
    scheduler.load_state_dict(saved['scheduler']);scaler.load_state_dict(saved['scaler'])
    random.setstate(saved['python_rng']);np.random.set_state(saved['numpy_rng'])
    torch.set_rng_state(saved['torch_rng']);torch.cuda.set_rng_state_all(saved['cuda_rng'])
    start_epoch=saved['epoch'];completed_steps=saved['completed_steps'];history=saved['history']
    best_cer=saved['best_cer'];best_epoch=saved['best_epoch']
print({'steps_per_epoch':steps_per_epoch,'total_steps':total_steps,'warmup_steps':warmup_steps,
 'start_epoch':start_epoch,'fingerprint':fingerprint})
def batch_from(indices):
    x=torch.from_numpy(np.stack([train_features[i] for i in indices])).to(DEVICE,dtype=torch.float32)
    packed=processor.tokenizer.pad({'input_ids':[train_targets[i] for i in indices]},padding=True,return_tensors='pt')
    labels=packed['input_ids'].masked_fill(packed['attention_mask'].ne(1),-100)
    for row,i in enumerate(indices):assert labels[row,len(train_targets[i])-1].item()==processor.tokenizer.eos_token_id
    return x,labels.to(DEVICE)


## 4. Full-validation corpus CER

Each checkpoint generates on all 732 validation clips in FP32 with the same B0 policy. Raw references/hypotheses and failed outputs are saved privately. The headline score sums per-clip S/D/I and divides by all 64,137 NFD reference code points. No test input is mounted.


In [ ]:
def edit_counts(reference, hypothesis):
    """Return one optimal code-point edit decomposition (S, D, I)."""
    ref = list(reference)
    hyp = list(hypothesis)
    rows, cols = len(ref) + 1, len(hyp) + 1
    distance = [[0] * cols for _ in range(rows)]
    for i in range(rows):
        distance[i][0] = i
    for j in range(cols):
        distance[0][j] = j
    for i in range(1, rows):
        for j in range(1, cols):
            substitution_cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            distance[i][j] = min(
                distance[i - 1][j] + 1,
                distance[i][j - 1] + 1,
                distance[i - 1][j - 1] + substitution_cost,
            )
    substitutions = deletions = insertions = 0
    i, j = len(ref), len(hyp)
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1] and distance[i][j] == distance[i - 1][j - 1]:
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and distance[i][j] == distance[i - 1][j - 1] + 1:
            substitutions += 1
            i -= 1
            j -= 1
        elif i > 0 and distance[i][j] == distance[i - 1][j] + 1:
            deletions += 1
            i -= 1
        elif j > 0 and distance[i][j] == distance[i][j - 1] + 1:
            insertions += 1
            j -= 1
        else:
            raise RuntimeError("Could not backtrack the character edit-distance matrix.")
    return substitutions, deletions, insertions


def evaluate(epoch):
    model.eval();model.config.use_cache=True
    totals={'S':0,'D':0,'I':0,'reference_codepoints':0,'decode_errors':0,'empty_predictions':0,
            'eos_terminated':0,'cap_without_eos':0,'replacement_character_outputs':0}
    private=OUT/f'validation_epoch_{epoch:02d}_private.jsonl'
    begun=time.perf_counter()
    with private.open('w',encoding='utf-8') as stream:
        for row in validation:
            ref=normalized(row['reference']);assert ref==row['reference']
            hyp='';status='ok';error='';ids=[]
            try:
                waveform=read_audio(VAL_ROOT,row)
                feature=processor.feature_extractor(waveform,sampling_rate=16000,return_tensors='pt').input_features.to(DEVICE,dtype=torch.float32)
                with torch.inference_mode():
                    output=model.generate(input_features=feature,generation_config=copy.deepcopy(generation_config))
                ids=output.sequences[0].detach().cpu().tolist()
                hyp=normalized(processor.tokenizer.decode(ids,skip_special_tokens=True))
                if not hyp:status='empty_prediction'
            except Exception as exc:
                status='decode_error';error=f'{type(exc).__name__}: {str(exc)[:300]}'
                if isinstance(exc,torch.cuda.OutOfMemoryError):torch.cuda.empty_cache()
            sub,dele,ins=edit_counts(ref,hyp)
            assert sub+dele+ins>=0
            for key,value in [('S',sub),('D',dele),('I',ins),('reference_codepoints',len(ref))]:totals[key]+=value
            totals['decode_errors']+=status=='decode_error';totals['empty_predictions']+=status=='empty_prediction'
            totals['eos_terminated']+=bool(ids and ids[-1]==processor.tokenizer.eos_token_id)
            totals['cap_without_eos']+=bool(len(ids)>=448 and (not ids or ids[-1]!=processor.tokenizer.eos_token_id))
            totals['replacement_character_outputs']+='\ufffd' in hyp
            stream.write(json.dumps({'sample_id':row['sample_id'],'reference':ref,'hypothesis':hyp,
              'S':sub,'D':dele,'I':ins,'status':status,'error':error,'duration_seconds':float(row['duration_seconds'])},ensure_ascii=False)+'\n')
    assert totals['reference_codepoints']==64137 and len(validation)==732
    totals['clips']=732;totals['hours']=sum(float(r['duration_seconds']) for r in validation)/3600
    totals['edit_count']=totals['S']+totals['D']+totals['I']
    totals['corpus_cer']=totals['edit_count']/totals['reference_codepoints']
    totals['elapsed_seconds']=time.perf_counter()-begun
    model.train();model.config.use_cache=False
    return totals


## 5. Train, evaluate, select and save

Each epoch shuffles only training membership, accumulates exactly one effective update per group (including final partial group), verifies finite AMP gradients and real optimizer progress, evaluates full validation, and saves checkpoint/training state. The minimum validation CER checkpoint is selected; test is untouched. Training loss and pilot CER are separate quantities.


In [ ]:
train_started=time.perf_counter();torch.cuda.reset_peak_memory_stats(DEVICE)
for epoch in range(start_epoch+1,MAX_EPOCHS+1):
    indices=list(range(len(train)));random.shuffle(indices)
    micro_groups=[indices[i:i+MICRO_BATCH] for i in range(0,len(indices),MICRO_BATCH)]
    epoch_losses=[];epoch_steps=0;amp_retries=0
    for begin in range(0,len(micro_groups),ACCUMULATION):
        groups=micro_groups[begin:begin+ACCUMULATION]
        # Reuse the same effective batch if GradScaler detects overflow.
        for attempt in range(8):
            optimizer.zero_grad(set_to_none=True);group_losses=[]
            for ids in groups:
                x,y=batch_from(ids)
                with torch.autocast('cuda',dtype=torch.float16):loss=model(input_features=x,labels=y).loss
                if not torch.isfinite(loss).item():raise RuntimeError('Nonfinite forward loss; stop pilot.')
                group_losses.append(float(loss.detach()))
                scaler.scale(loss/len(groups)).backward()
            scaler.unscale_(optimizer)
            nonfinite=sum(p.grad is not None and not torch.isfinite(p.grad).all().item() for p in model.parameters())
            before_scale=scaler.get_scale()
            if nonfinite:
                scaler.step(optimizer)  # GradScaler skips an optimizer step with nonfinite gradients.
                scaler.update();amp_retries+=1;continue
            grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True,foreach=False)
            if float(grad_norm)<=0:raise RuntimeError('Zero gradient norm; stop pilot.')
            scaler.step(optimizer);scaler.update()
            if scaler.get_scale()<before_scale:
                amp_retries+=1;continue
            scheduler.step();completed_steps+=1;epoch_steps+=1;epoch_losses.extend(group_losses)
            break
        else:raise RuntimeError('No finite AMP update after 8 retries; stop pilot.')
    assert epoch_steps==steps_per_epoch
    train_mib=torch.cuda.max_memory_allocated(DEVICE)/1024**2
    validation_metrics=evaluate(epoch)
    improved=validation_metrics['corpus_cer']<best_cer
    if improved:best_cer=validation_metrics['corpus_cer'];best_epoch=epoch
    record={'epoch':epoch,'updates':epoch_steps,'cumulative_updates':completed_steps,
      'training_loss_mean_of_microbatches':sum(epoch_losses)/len(epoch_losses),
      'amp_retries':amp_retries,'train_peak_allocated_mib':train_mib,
      'validation':validation_metrics}
    history.append(record)
    checkpoint=OUT/f'epoch_{epoch:02d}';checkpoint.mkdir(exist_ok=True)
    model.save_pretrained(checkpoint/'model',safe_serialization=True)
    state={'fingerprint':fingerprint,'epoch':epoch,'completed_steps':completed_steps,
      'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
      'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
      'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
      'history':history,'best_cer':best_cer,'best_epoch':best_epoch}
    torch.save(state,checkpoint/'training_state.pt')
    (OUT/'aggregate_history.json').write_text(json.dumps(history,indent=2)+'\n')
    print({'epoch':epoch,'updates':epoch_steps,'validation_cer_percent':100*validation_metrics['corpus_cer'],
      'best_epoch':best_epoch,'amp_retries':amp_retries,'train_peak_allocated_mib':train_mib,
      'validation_seconds':validation_metrics['elapsed_seconds']})
report={'experiment':'P1','status':'complete','scope':'Phase 6B one-hour frozen-train pilot',
 'train_clips':len(train),'train_hours':TRAIN_RECORD['hours'],'validation_clips':732,'validation_hours':VAL_RECORD['hours'],
 'seed':SEED,'model_id':MODEL_ID,'model_revision':REVISION,'versions':{'python':platform.python_version(),
 'torch':torch.__version__,'transformers':transformers.__version__,'tokenizers':tokenizers.__version__,
 'hub':huggingface_hub.__version__,'safetensors':safetensors.__version__},
 'gpu':torch.cuda.get_device_name(0),'micro_batch':MICRO_BATCH,'accumulation':ACCUMULATION,
 'max_epochs':MAX_EPOCHS,'completed_steps':completed_steps,'warmup_steps':warmup_steps,
 'train_membership_sha256':TRAIN_RECORD['train_membership_sha256'],
 'validation_manifest_sha256':VAL_RECORD['manifest_sha256'],'run_fingerprint':fingerprint,
 'baseline_B0_cer':1.031292389728238,'best_validation_cer':best_cer,'best_epoch':best_epoch,
 'total_wall_seconds':time.perf_counter()-train_started,'peak_allocated_mib':torch.cuda.max_memory_allocated(DEVICE)/1024**2,
 'peak_reserved_mib':torch.cuda.max_memory_reserved(DEVICE)/1024**2,'training_clip_exposures':len(train)*MAX_EPOCHS,
 'test_data_used':False,'history':history,
 'note':'Training loss is diagnostic; model selection uses full-validation NFD code-point CER. Checkpoints and predictions are private.'}
(OUT/'P1_report_private.json').write_text(json.dumps(report,indent=2)+'\n')
print('P1 pilot complete. Best epoch:',best_epoch,'validation CER:',100*best_cer,'%. Private output:',OUT)


## 6. Aggregate plots from the completed run

Run only after the training cell finishes. These figures use saved aggregate metrics and the best epoch's private validation JSONL; they do **not** run inference again. No clip IDs or transcript text appear in the figures. A resumed run needs the earlier best epoch's JSONL copied into the current output directory if that epoch remains best.


In [ ]:
from pathlib import Path
import json, math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

result_root=Path('/kaggle/working/phase6b_P1_private')
report_path=result_root/'P1_report_private.json'
assert report_path.is_file(), 'Finish Phase 6B training before plotting.'
run_report=json.loads(report_path.read_text())
assert run_report['status']=='complete' and run_report['test_data_used'] is False
run_history=run_report['history']
assert len(run_history)==run_report['max_epochs'] and all(r['validation']['clips']==732 for r in run_history)
best=int(run_report['best_epoch'])
prediction_path=result_root/f'validation_epoch_{best:02d}_private.jsonl'
assert prediction_path.is_file(), 'Best-epoch private predictions are missing; restore them from the earlier session if this is a resume.'
predictions=[json.loads(line) for line in prediction_path.read_text(encoding='utf-8').splitlines() if line]
assert len(predictions)==732 and sum(len(r['reference']) for r in predictions)==64137
assert len({r['sample_id'] for r in predictions})==732
assert sum(r['S']+r['D']+r['I'] for r in predictions)==run_history[best-1]['validation']['edit_count']

plot_dir=result_root/'plots';plot_dir.mkdir(exist_ok=True)
epochs=[r['epoch'] for r in run_history]
cer=[100*r['validation']['corpus_cer'] for r in run_history]
losses=[r['training_loss_mean_of_microbatches'] for r in run_history]
fig,ax=plt.subplots(2,2,figsize=(12,9),constrained_layout=True)
ax[0,0].plot(epochs,cer,'o-',label='P1 validation')
ax[0,0].axhline(100*run_report['baseline_B0_cer'],color='gray',linestyle='--',label='B0 zero-shot')
ax[0,0].set(title='Full-validation corpus CER',xlabel='Epoch',ylabel='CER (%)',xticks=epochs)
ax[0,0].legend()
ax[0,1].plot(epochs,losses,'o-',color='#b15d1a')
ax[0,1].set(title='Training loss (diagnostic)',xlabel='Epoch',ylabel='Mean micro-batch loss',xticks=epochs)
colors={'S':'#4c78a8','D':'#f58518','I':'#e45756'}
bottom=[0]*len(epochs)
for key,label in [('S','Substitutions'),('D','Deletions'),('I','Insertions')]:
    values=[r['validation'][key] for r in run_history]
    ax[1,0].bar(epochs,values,bottom=bottom,color=colors[key],label=label)
    bottom=[a+b for a,b in zip(bottom,values)]
ax[1,0].set(title='Validation edit counts (732 clips each epoch)',xlabel='Epoch',ylabel='Edits',xticks=epochs)
ax[1,0].legend()
flags=[('cap_without_eos','448-token cap without EOS'),('replacement_character_outputs','U+FFFD output'),('decode_errors','Decode error'),('empty_predictions','Empty prediction')]
for key,label in flags:
    ax[1,1].plot(epochs,[r['validation'][key] for r in run_history],marker='o',label=label)
ax[1,1].set(title='Validation generation diagnostics',xlabel='Epoch',ylabel='Clips',xticks=epochs)
ax[1,1].legend(fontsize=8)
fig.savefig(plot_dir/'phase6b_learning_and_validation.png',dpi=170)
plt.close(fig)

bins=[(0.5,5),(5,10),(10,20),(20,30)]
by_duration=[]
for lo,hi in bins:
    part=[r for r in predictions if lo<=float(r['duration_seconds'])<hi]
    chars=sum(len(r['reference']) for r in part)
    edits=sum(r['S']+r['D']+r['I'] for r in part)
    by_duration.append({'bin':f'{lo:g}–{hi:g} s','clips':len(part),'reference_codepoints':chars,'edits':edits,'cer_percent':100*edits/chars if chars else None})
assert sum(x['clips'] for x in by_duration)==732 and sum(x['reference_codepoints'] for x in by_duration)==64137
fig,axis=plt.subplots(figsize=(8,4.5),layout='constrained')
values=[x['cer_percent'] if x['cer_percent'] is not None else 0 for x in by_duration]
bars=axis.bar([x['bin'] for x in by_duration],values,color='#4c78a8')
for bar,item in zip(bars,by_duration):
    axis.annotate(f"n={item['clips']}",(bar.get_x()+bar.get_width()/2,bar.get_height()),ha='center',va='bottom')
axis.set(title=f'Best epoch {best}: CER by clip duration',xlabel='Duration',ylabel='Corpus CER within bin (%)')
fig.savefig(plot_dir/'phase6b_best_cer_by_duration.png',dpi=170)
plt.close(fig)
plot_data={'status':'complete','validation_clips_per_epoch':732,'reference_codepoints_per_epoch':64137,
           'epochs':epochs,'cer_percent':cer,'training_loss':losses,'best_epoch':best,
           'best_cer_by_duration':by_duration,'test_data_used':False}
(plot_dir/'aggregate_plot_data.json').write_text(json.dumps(plot_data,indent=2)+'\n')
print('Saved aggregate plots:',*[str(p) for p in sorted(plot_dir.glob('*.png'))],sep='\n')


## 7. ZIP the private results

This small ZIP includes the report, metrics, plots and per-clip validation predictions. **Keep it private**: JSONL files contain IDs and transcript text. Model weights and optimizer state stay in the epoch checkpoint directories; the optional next cell can package the best checkpoint separately. Download from Kaggle's **Output** tab if a notebook link fails.


In [ ]:
import hashlib, zipfile
from IPython.display import FileLink, display

result_root=Path('/kaggle/working/phase6b_P1_private')
report=json.loads((result_root/'P1_report_private.json').read_text())
assert report['status']=='complete' and report['test_data_used'] is False
expected=[result_root/'P1_report_private.json',result_root/'aggregate_history.json',
          result_root/'plots'/'aggregate_plot_data.json',
          result_root/'plots'/'phase6b_learning_and_validation.png',
          result_root/'plots'/'phase6b_best_cer_by_duration.png']
expected += [result_root/f'validation_epoch_{epoch:02d}_private.jsonl' for epoch in range(1,report['max_epochs']+1)]
missing=[str(path) for path in expected if not path.is_file()]
assert not missing, f'Missing run files (restore earlier epoch outputs after a resume): {missing}'
zip_path=Path('/kaggle/working/phase6b_P1_results_private.zip')
checks=[]
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    for path in expected:
        arc='phase6b_P1_results/'+path.relative_to(result_root).as_posix()
        data=path.read_bytes()
        z.writestr(arc,data)
        checks.append(f"{hashlib.sha256(data).hexdigest()}  {arc}")
    z.writestr('phase6b_P1_results/SHA256SUMS.txt','\n'.join(checks)+'\n')
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
    print('Private result files:',len(z.namelist()),'ZIP MiB:',round(zip_path.stat().st_size/1024**2,2))
print('SHA-256:',hashlib.sha256(zip_path.read_bytes()).hexdigest())
print('Download from Kaggle Output:',zip_path)
display(FileLink(str(zip_path)))


## 8. Optional separate best-checkpoint ZIP

Use this only if you need a portable training checkpoint. It may be hundreds of MiB or larger and can be harder to download through the browser. The checkpoint remains available under `/kaggle/working/phase6b_P1_private/epoch_XX/` even if you skip this ZIP.


In [ ]:
MAKE_BEST_CHECKPOINT_ZIP=False
if MAKE_BEST_CHECKPOINT_ZIP:
    best=int(json.loads((result_root/'P1_report_private.json').read_text())['best_epoch'])
    checkpoint_dir=result_root/f'epoch_{best:02d}'
    assert (checkpoint_dir/'training_state.pt').is_file() and (checkpoint_dir/'model'/'config.json').is_file(), 'Restore best checkpoint from the earlier Kaggle session first.'
    checkpoint_zip=Path('/kaggle/working/phase6b_P1_best_checkpoint_private.zip')
    with zipfile.ZipFile(checkpoint_zip,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
        for path in sorted(checkpoint_dir.rglob('*')):
            if path.is_file():z.write(path,arcname='phase6b_P1_best_checkpoint/'+path.relative_to(checkpoint_dir).as_posix())
    with zipfile.ZipFile(checkpoint_zip) as z:assert z.testzip() is None
    print('Best checkpoint ZIP MiB:',round(checkpoint_zip.stat().st_size/1024**2,2),checkpoint_zip)
    display(FileLink(str(checkpoint_zip)))
else:
    print('Checkpoint ZIP skipped; set MAKE_BEST_CHECKPOINT_ZIP=True only if needed.')
